In [ ]:
# Download RWF-2000 with KaggleHub; recover from the archive if automatic unzip hits the dataset's long UTF-8 filenames.
from pathlib import Path
from zipfile import ZipFile
import importlib.util, subprocess, sys
if importlib.util.find_spec('kagglehub') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
import kagglehub
try:
    path = kagglehub.dataset_download('vulamnguyen/rwf2000')
    print('Path to dataset files:', path)
except OSError as error:
    print('KaggleHub auto-extraction hit a filename encoding issue; using its downloaded archive:', error)
    archive = Path('/root/.cache/kagglehub/datasets/vulamnguyen/rwf2000/1.archive')
    assert archive.is_file(), f'KaggleHub archive not found: {archive}'
    fallback = Path('/content/kagglehub_rwf2000')
    fallback.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive, metadata_encoding='utf-8') as zf:
        members = [item for item in zf.infolist() if not item.is_dir() and item.filename.startswith('RWF-2000/')]
        assert len(members) == 2000, f'Expected 2000 videos in archive, found {len(members)}'
        zf.extractall(fallback, members=members)
    path = str(fallback)
    print('Extracted KaggleHub archive to:', path)
print('Dataset source:', path)

Using Colab cache for faster access to the 'rwf2000' dataset.
Path to dataset files: /kaggle/input/rwf2000


In [ ]:
# Keep KaggleHub videos on Colab storage; never copy them to Drive.
from pathlib import Path
kaggle_root = Path(path)
dataset_source = kaggle_root / 'RWF-2000'
if not (dataset_source/'train').is_dir() and (kaggle_root/'train').is_dir():
    dataset_source = kaggle_root
assert (dataset_source/'train').is_dir() and (dataset_source/'val').is_dir(), f'Expected train/val under KaggleHub path: {kaggle_root}'
for split in ('train', 'val'):
    print(split, {folder.name: sum(1 for f in folder.iterdir() if f.is_file()) for folder in (dataset_source/split).iterdir() if folder.is_dir()})
print('KaggleHub data remains on Colab local storage:', dataset_source)

Copying files from /kaggle/input/rwf2000 to /content/drive/MyDrive/Vi-SAFE-3B42/...
Copy completed successfully!


In [ ]:
import os

target_dir = '/content/drive/MyDrive/Vi-SAFE-3B42'
output_file = os.path.join(target_dir, 'folder_structure.txt')

if os.path.exists(target_dir):
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(f"Listing contents of: {target_dir}\n\n")
        for root, dirs, files in os.walk(target_dir):
            level = root.replace(target_dir, '').count(os.sep)
            indent = ' ' * 4 * level
            f.write(f"{indent}[Dir] {os.path.basename(root)}/\n")
            sub_indent = ' ' * 4 * (level + 1)
            for file in files:
                if file != 'folder_structure.txt':
                    f.write(f"{sub_indent}[File] {file}\n")
    print(f"Successfully saved the folder structure to {output_file}")
else:
    print(f"Directory {target_dir} does not exist. Please make sure Google Drive is mounted.")

Successfully saved the folder structure to /content/drive/MyDrive/Vi-SAFE-3B42/folder_structure.txt


In [ ]:
%cd /content/drive/MyDrive/Vi-SAFE-3B42

!sed -n '1,240p' app/encoder_yolo.py
!grep -RniE "yolov8|\.pt|weights|checkpoint|pretrained" \
    app app_simple README.md requirements.txt

/content/drive/MyDrive/Vi-SAFE-3B42
import torch
from torch import nn
import torch.nn.functional as F
from ultralytics import YOLO


class BboxTransformerAggregator(nn.Module):

    def __init__(
        self,
        input_dim=13,
        hidden_dim=16,
        aggregator_out_dim=128,
        num_heads=2,
        use_cls_token=False
    ):
        """
        :param input_dim:        dimension of the per-bbox input (e.g. 5 + class_embed_dim)
        :param hidden_dim:       dimension for the transformer's internal d_model
                                 (must be divisible by num_heads)
        :param aggregator_out_dim: dimension of final output (e.g. 128)
        :param num_heads:        multi-head attention heads (hidden_dim must be divisible by this)
        :param use_cls_token:    whether to add a learnable [CLS] token for pooling
        """
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.aggregator_out_dim = aggre

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Vi-SAFE-3B42

/content/drive/MyDrive/Vi-SAFE-3B42


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 7.5 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import os

model = YOLO("yolov8n.pt")

print("YOLO weights loaded successfully!")
print("Weights path:", model.ckpt_path if hasattr(model, "ckpt_path") else "Loaded")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLO weights loaded successfully!
Weights path: yolov8n.pt


In [ ]:
import os

base = "/content/drive/MyDrive/Vi-SAFE-3B42/RWF-2000"

for split in ["train", "val"]:
    print(f"\n{split.upper()}")
    split_path = os.path.join(base, split)

    for label in sorted(os.listdir(split_path)):
        folder = os.path.join(split_path, label)

        if os.path.isdir(folder):
            videos = [
                f for f in os.listdir(folder)
                if f.lower().endswith((".avi", ".mp4", ".mov", ".mkv"))
            ]
            print(f"{label}: {len(videos)} videos")


TRAIN
Fight: 800 videos
NonFight: 800 videos

VAL
Fight: 200 videos
NonFight: 200 videos


In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/Vi-SAFE-3B42")

for name in ["README.md", "app_simple/main.py", "app/main.py",
             "app_simple/dataset_rwf2000.py", "app_simple/yolo_tsn.py"]:
    path = root / name
    print(f"\n{'=' * 20} {name} {'=' * 20}")
    if path.exists():
        lines = path.read_text(errors="ignore").splitlines()
        for i, line in enumerate(lines[:180], 1):
            print(f"{i:3}: {line}")
    else:
        print("FILE NOT FOUND")


==================== README.md ====================
  1: # Vi-SAFE
  2: 
  3: A video violence detection system combining YOLOv8 object detection with Temporal Segment Networks (TSN) for the RWF-2000 dataset.
  4: 
  5: ## Quick Start
  6: 
  7: ### Installation
  8: ```bash
  9: pip install -r requirements.txt
 10: ```
 11: 
 12: ### Download Model Weights
 13: 
 14: https://github.com/ultralytics/assets/releases
 15: 
 16: ### Data Structure
 17: ```
 18: data/
 19: ├── train/
 20: │   ├── Fight/      # Violence videos
 21: │   └── NonFight/   # Normal videos  
 22: └── val/
 23:     ├── Fight/
 24:     └── NonFight/
 25: ```
 26: 
 27: ### Training
 28: ```bash
 29: # Simple version
 30: cd app_simple
 31: python main.py --rwf2000_root /path/to/data --epochs 50
 32: 
 33: # Full version with advanced features
 34: cd app
 35: python main.py --rwf2000_root /path/to/data --epochs 100
 36: ```
 37: 
 38: ## Model Architecture
 39: 
 40: - **YOLOv8**: Object detection and spatial encod

In [ ]:
%cd /content/drive/MyDrive/Vi-SAFE-3B42
!pip install -r requirements.txt

/content/drive/MyDrive/Vi-SAFE-3B42
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 51.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


# Vi-SAFE + RWF-2000 Violence Detection Pipeline

This notebook stages the repository and videos on Colab local storage for faster reads, checks dependencies and code paths, runs a small data/model smoke test, trains the paper implementation, and copies checkpoints, metrics, and logs to Google Drive.

In [ ]:
# CELL: Verify runtime, paths, and local disk before staging
import os, shutil, sys, importlib.util, subprocess
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Vi-SAFE-3B42')
LOCAL_ROOT = Path('/content/Vi-SAFE-3B42')
DRIVE_DATA = DRIVE_ROOT / 'RWF-2000'
LOCAL_DATA = LOCAL_ROOT / 'RWF-2000'
DRIVE_OUTPUT = DRIVE_ROOT / 'outputs'

assert DRIVE_ROOT.exists(), f'Repository not found: {DRIVE_ROOT}; confirm MyDrive is mounted.'
assert DRIVE_DATA.exists(), f'Dataset not found: {DRIVE_DATA}'
free_gb = shutil.disk_usage('/content').free / 1024**3
print(f'Python: {sys.version.split()[0]} | free /content: {free_gb:.1f} GiB')
print('Drive source:', DRIVE_ROOT)
print('Installed modules:', {m: bool(importlib.util.find_spec(m)) for m in ['torch','torchvision','cv2','ultralytics','mmengine','mmaction','wandb','decord']})
try:
 import torch
 print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| CUDA:', torch.version.cuda)
 if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except Exception as e: print('Torch import error:', repr(e))
print('Drive dataset size GiB:', round(sum(p.stat().st_size for p in DRIVE_DATA.rglob('*') if p.is_file())/1024**3, 2))

Python: 3.13.16 | free /content: 70.1 GiB
Drive source: /content/drive/MyDrive/Vi-SAFE-3B42
Installed modules: {'torch': True, 'torchvision': True, 'cv2': True, 'ultralytics': False, 'mmengine': False, 'mmaction': False, 'wandb': True, 'decord': False}
Torch: 2.11.0+cu130 | CUDA available: True | CUDA: 13.0
GPU: Tesla T4
Drive dataset size GiB: 11.83


In [ ]:
# CELL: Stage Vi-SAFE app code and Kagglehub videos in /content
from pathlib import Path
import shutil

kaggle_root = Path(path)
dataset_source = kaggle_root / 'RWF-2000'
assert dataset_source.is_dir(), f'Dataset missing under {kaggle_root}'
print('Dataset splits:', {s: [p.name for p in (dataset_source/s).iterdir()] for s in ['train','val']})
required_bytes = sum(p.stat().st_size for p in dataset_source.rglob('*') if p.is_file())
assert shutil.disk_usage('/content').free > required_bytes + 5*1024**3, 'Insufficient /content disk space.'
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
for name in ['app','app_simple']:
    src, dst = DRIVE_ROOT/name, LOCAL_ROOT/name
    if dst.exists(): shutil.rmtree(dst)
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns('__pycache__','*.pyc'))
for name in ['README.md','requirements.txt','yolov8n.pt']:
    src = DRIVE_ROOT/name
    if src.exists(): shutil.copy2(src, LOCAL_ROOT/name)
if LOCAL_DATA.exists(): shutil.rmtree(LOCAL_DATA)
shutil.copytree(dataset_source, LOCAL_DATA)
print('Code and data staged at:', LOCAL_ROOT)
print('Local data size GiB:', round(sum(p.stat().st_size for p in LOCAL_DATA.rglob('*') if p.is_file())/1024**3,2))
print('Free /content GiB:', round(shutil.disk_usage('/content').free/1024**3,1))

Dataset splits: {'train': ['Fight', 'NonFight'], 'val': ['Fight', 'NonFight']}
Code and data staged at: /content/Vi-SAFE-3B42
Local data size GiB: 11.83
Free /content GiB: 58.3


In [ ]:
# CELL: Check dependencies and resolve the TSN config from the Drive checkout
import sys, importlib.util
from pathlib import Path
%cd /content/Vi-SAFE-3B42
MM_ROOT = DRIVE_ROOT / 'mmaction2'
MM_CONFIGS = MM_ROOT / 'configs'
sys.path.insert(0, str(MM_ROOT))
print('CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
for module in ['torch','torchvision','cv2','ultralytics','mmengine','mmcv','mmaction','wandb','decord']:
    print(module, bool(importlib.util.find_spec(module)))
config_candidates = list((MM_CONFIGS/'recognition'/'tsn').glob('*r50*kinetics400*rgb.py'))
for cfg in config_candidates[:10]: print('TSN config:', cfg)
assert config_candidates, f'No TSN R50 config found under {MM_CONFIGS}'
CONFIG_PATH = next((p for p in config_candidates if 'imagenet-pretrained' in p.name), config_candidates[0])
print('Selected config:', CONFIG_PATH)

/content/Vi-SAFE-3B42
CUDA: True | GPU: Tesla T4
torch True
torchvision True
cv2 True
ultralytics False
mmengine False
mmcv False
mmaction True
wandb True
decord False
TSN config: /content/drive/MyDrive/Vi-SAFE-3B42/mmaction2/configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb.py
TSN config: /content/drive/MyDrive/Vi-SAFE-3B42/mmaction2/configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x5-100e_kinetics400-rgb.py
TSN config: /content/drive/MyDrive/Vi-SAFE-3B42/mmaction2/configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x8-100e_kinetics400-rgb.py
TSN config: /content/drive/MyDrive/Vi-SAFE-3B42/mmaction2/configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-dense-1x1x5-100e_kinetics400-rgb.py
Selected config: /content/drive/MyDrive/Vi-SAFE-3B42/mmaction2/configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb.py


In [ ]:
# CELL: Adapt the TSN classifier to the two RWF-2000 classes and inspect CLI options
from pathlib import Path
import re
%cd /content/Vi-SAFE-3B42
model_file = LOCAL_ROOT/'app'/'yolo_tsn.py'
source = model_file.read_text()
needle = 'self.tsn = MODELS.build(config.model)'
if needle in source and 'config.model.cls_head.num_classes = 2' not in source:
    source = source.replace(needle, "config.model.cls_head.num_classes = 2\n        self.tsn = MODELS.build(config.model)")
    model_file.write_text(source)
    print('Patched TSN classifier to num_classes=2 in', model_file)
else:
    print('Two-class patch already present or build line not found; inspect', model_file)
main_text = (LOCAL_ROOT/'app'/'main.py').read_text().splitlines()
print('Training CLI arguments:')
for line in main_text:
    if 'parser.add_argument' in line: print(line.strip())

/content/Vi-SAFE-3B42
Patched TSN classifier to num_classes=2 in /content/Vi-SAFE-3B42/app/yolo_tsn.py
Training CLI arguments:
parser.add_argument('--rwf2000_root', type=str,
parser.add_argument('--mmaction_cfg', type=str,
parser.add_argument('--yolo_weights', type=str,
parser.add_argument('--output_dir', type=str, default="./checkpoints",
parser.add_argument('--epochs', type=int, default=100, help="训练轮数")
parser.add_argument('--batch_size', type=int, default=8, help="batch size")
parser.add_argument('--num_segments', type=int, default=8, help="TSN 帧段数")
parser.add_argument('--lr', type=float, default=1e-3, help="初始学习率")
parser.add_argument('--weight_decay', type=float, default=1e-5, help="权重衰减系数")
parser.add_argument('--train_all_params', default=True, action='store_true',
parser.add_argument('--freeze_yolo', action='store_true',
parser.add_argument('--freeze_tsn_backbone', action='store_true',
parser.add_argument('--seed', type=int, default=42, help="随机种子")
parser.add_argument('--dev

In [ ]:
# CELL: Install only missing project dependencies; preserve Colab's CUDA PyTorch
%pip install -q "ultralytics>=8.0.105,<9" "mmengine>=0.10,<1" "mmcv-lite>=2.0,<2.2"

import torch, ultralytics, mmengine, mmcv
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
print('ultralytics', ultralytics.__version__, '| mmengine', mmengine.__version__, '| mmcv', mmcv.__version__)
assert torch.cuda.is_available(), 'CUDA is required before training.'

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
torch 2.11.0+cu130 | CUDA True
ultralytics 8.4.185 | mmengine 0.10.7 | mmcv 2.1.0


In [ ]:
# CELL: Reload the patched YOLOv8 encoder and verify one fused Vi-SAFE prediction
import torch, sys, importlib
sys.path.insert(0, str(LOCAL_ROOT))
from mmengine.registry import DefaultScope
if DefaultScope.get_current_instance() is None:
    DefaultScope.get_instance('mmaction', scope_name='mmaction')
import app.encoder_yolo as yolo_encoder_module
yolo_encoder_module = importlib.reload(yolo_encoder_module)
YOLOEncoder = yolo_encoder_module.YOLOEncoder
from app.dataset_rwf2000 import RWF2000Dataset
from app.yolo_tsn import TSNWithYOLO
device = 'cuda' if torch.cuda.is_available() else 'cpu'
smoke_val = RWF2000Dataset(root_dir=str(LOCAL_DATA), mode='val', num_segments=8)
smoke_frames, smoke_label = smoke_val[0]
yolo_encoder = YOLOEncoder(yolo_weights_path=str(LOCAL_ROOT/'yolov8n.pt'), device=device, freeze_yolo=True)
print('Registered encoder children:', list(yolo_encoder._modules))
assert 'yolo_model' not in yolo_encoder._modules, 'YOLO wrapper must not receive PyTorch recursive train/eval calls.'
yolo_encoder = yolo_encoder.to(device).eval()
fusion_model = TSNWithYOLO(mmaction_cfg_path=str(CONFIG_PATH), device=device, freeze_tsn_backbone=True).to(device).eval()
with torch.inference_mode():
    yolo_features = yolo_encoder(smoke_frames.to(device))
    logits = fusion_model(smoke_frames.unsqueeze(0).to(device), yolo_features.unsqueeze(0))
print('validation label:', int(smoke_label), '| YOLO features:', tuple(yolo_features.shape), '| logits:', tuple(logits.shape))
assert tuple(logits.shape) == (1, 2), f'Expected 1x2 violence logits, got {tuple(logits.shape)}'
print('Fused model smoke test passed.')

[RWF2000Dataset] Mode: val
  Found 200 fight videos in: /content/Vi-SAFE-3B42/RWF-2000/val/Fight
  Found 200 nonfight videos in: /content/Vi-SAFE-3B42/RWF-2000/val/NonFight
  => Total 400 videos.

Registered encoder children: ['class_embedding', 'mlp', 'frame_residual_cnn']
validation label: 0 | YOLO features: (8, 128) | logits: (1, 2)
Fused model smoke test passed.


In [ ]:
# CELL: Validate local dataset structure and decode one clip before model training
%cd /content/Vi-SAFE-3B42
import os
from app.dataset_rwf2000 import RWF2000Dataset

for split in ['train', 'val']:
    for label in ['Fight', 'NonFight']:
        folder = LOCAL_DATA / split / label
        videos = [p for p in folder.iterdir() if p.suffix.lower() in {'.avi','.mp4','.mov','.mkv'}]
        print(f'{split}/{label}: {len(videos)} videos')

train_set = RWF2000Dataset(root_dir=str(LOCAL_DATA), mode='train', num_segments=8)
frames, label = train_set[0]
print('Decoded clip shape:', getattr(frames, 'shape', None), '| label:', label)
assert len(train_set) == 1600, f'Expected 1600 train videos, found {len(train_set)}'
assert frames is not None and len(frames) == 8, 'Expected 8 sampled frames.'
print('Dataset smoke check passed.')

/content/Vi-SAFE-3B42
train/Fight: 800 videos
train/NonFight: 800 videos
val/Fight: 200 videos
val/NonFight: 200 videos
[RWF2000Dataset] Mode: train
  Found 800 fight videos in: /content/Vi-SAFE-3B42/RWF-2000/train/Fight
  Found 800 nonfight videos in: /content/Vi-SAFE-3B42/RWF-2000/train/NonFight
  => Total 1600 videos.

Decoded clip shape: torch.Size([8, 3, 224, 224]) | label: tensor(0)
Dataset smoke check passed.


In [ ]:
EPOCHS = 14
BATCH_SIZE = 2
NUM_SEGMENTS = 8
NUM_WORKERS = 2
LEARNING_RATE = 1e-3
RUN_NAME = 'visafe-rwf2000-temporal-attention-14ep'
EARLY_STOP_PATIENCE = 14
TEMPORAL_ATTENTION = True
RESUME_CHECKPOINT = None
RUN_TRAINING = True
print('Training enabled:', RUN_TRAINING)
print('Temporal attention enabled:', TEMPORAL_ATTENTION)
print('Resume checkpoint:', RESUME_CHECKPOINT)
print('Checkpoint/output directory:', DRIVE_OUTPUT)
print('Videos are read from local Colab storage:', LOCAL_DATA)
print('Run config:', {'epochs': EPOCHS, 'batch': BATCH_SIZE, 'segments': NUM_SEGMENTS, 'workers': NUM_WORKERS})Training command:

Training enabled: True
Temporal attention enabled: True
Resume checkpoint: None
Checkpoint/output directory: /content/drive/MyDrive/Shared/Vi-SAFE-3B42/outputs/rwf2000_visafe_temporal_attention
Videos are read from local Colab storage: /content/Vi-SAFE-3B42/RWF-2000
Run config: {'epochs': 14, 'batch': 2, 'segments': 8, 'workers': 2}


# CELL: Launch only when RUN_TRAINING is explicitly enabled; read local data and write backups to Drive.
from pathlib import Path
import os
import subprocess
import sys
main_path = LOCAL_ROOT / 'app' / 'main.py'
MM_LOCAL_ROOT = LOCAL_ROOT / 'mmaction2'
bootstrap_path = LOCAL_ROOT / '_colab_train_entry.py'
bootstrap_source = chr(10).join([
    'from mmengine.registry import DefaultScope',
    "DefaultScope.get_instance('mmaction', scope_name='mmaction')",
    'import runpy, sys',
    'sys.argv[0] = ' + repr(str(main_path)),
    'runpy.run_path(' + repr(str(main_path)) + ', run_name=' + repr('__main__') + ')',
    ''
])
compile(bootstrap_source, str(bootstrap_path), 'exec')
bootstrap_path.write_text(bootstrap_source, encoding='utf-8')
command = [
    sys.executable, '-u', str(bootstrap_path),
    '--rwf2000_root', str(LOCAL_DATA),
    '--mmaction_cfg', str(CONFIG_PATH),
    '--yolo_weights', str(LOCAL_ROOT / 'yolov8n.pt'),
    '--output_dir', str(DRIVE_OUTPUT),
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--num_segments', str(NUM_SEGMENTS),
    '--lr', str(LEARNING_RATE),
    '--weight_decay', '1e-5',
    '--num_workers', str(NUM_WORKERS),
    '--seed', '42',
    '--device', 'cuda',
    '--wandb_name', RUN_NAME,
    '--early_stop_patience', str(EARLY_STOP_PATIENCE),
    '--freeze_yolo',
    '--freeze_tsn_backbone',
]
if TEMPORAL_ATTENTION:
    command.append('--temporal_attention')
if RESUME_CHECKPOINT:
    command.extend(['--resume', str(RESUME_CHECKPOINT)])
run_env = os.environ.copy()
run_env['PYTHONPATH'] = os.pathsep.join([str(LOCAL_ROOT), str(MM_LOCAL_ROOT), run_env.get('PYTHONPATH', '')])
run_env['WANDB_MODE'] = 'disabled'
print('Training command:', ' '.join(command))
print('YOLOv8 weights:', LOCAL_ROOT / 'yolov8n.pt')
print('Video data (local):', LOCAL_DATA)
print('Drive outputs:', DRIVE_OUTPUT)
if RUN_TRAINING:
    log_path = DRIVE_OUTPUT / 'training.log'
    with log_path.open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=LOCAL_ROOT, env=run_env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
    print('Training finished. Checkpoints and training.log are saved in Drive.')
else:
    print('Training is disabled. Set RUN_TRAINING = True only when ready to start.')

In [ ]:
%pip install -q "transformers==4.47.1" "tokenizers>=0.21,<0.22"
%pip install -q --force-reinstall --no-deps mmcv-lite==2.1.0
# Train temporal-attention Vi-SAFE for the baseline's exact 14 completed epochs; persist checkpoints and logs to Drive.
from pathlib import Path
import os, subprocess, sys
LOCAL_ROOT = Path('/content/Vi-SAFE-3B42')
LOCAL_DATA = LOCAL_ROOT / 'RWF-2000'
DRIVE_OUTPUT = Path('/content/drive/MyDrive/Shared/Vi-SAFE-3B42/outputs/temporal_attention')
CONFIG_PATH = LOCAL_ROOT / 'mmaction2/configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb.py'
FULL_MMACTION = Path('/content/mmaction2_full_v1_2_0')
EPOCHS, BATCH_SIZE, NUM_SEGMENTS, NUM_WORKERS = 14, 2, 8, 2
LEARNING_RATE, EARLY_STOP_PATIENCE = 1e-3, 14
RUN_NAME = 'visafe-rwf2000-temporal-attention-14ep'
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
assert (LOCAL_DATA/'train').is_dir() and (LOCAL_DATA/'val').is_dir()
assert CONFIG_PATH.is_file() and (LOCAL_ROOT/'yolov8n.pt').is_file()
main_path = LOCAL_ROOT/'app/main.py'
bootstrap_path = LOCAL_ROOT/'_colab_train_entry.py'
bootstrap_source = '\n'.join(['from mmengine.registry import DefaultScope', "DefaultScope.get_instance('mmaction', scope_name='mmaction')", 'import runpy, sys', 'sys.argv[0] = '+repr(str(main_path)), 'runpy.run_path('+repr(str(main_path))+", run_name='__main__')", ''])
bootstrap_path.write_text(bootstrap_source, encoding='utf-8')
command = [sys.executable, '-u', str(bootstrap_path), '--rwf2000_root', str(LOCAL_DATA), '--mmaction_cfg', str(CONFIG_PATH), '--yolo_weights', str(LOCAL_ROOT/'yolov8n.pt'), '--output_dir', str(DRIVE_OUTPUT), '--epochs', str(EPOCHS), '--batch_size', str(BATCH_SIZE), '--num_segments', str(NUM_SEGMENTS), '--lr', str(LEARNING_RATE), '--weight_decay', '1e-5', '--num_workers', str(NUM_WORKERS), '--seed', '42', '--device', 'cuda', '--wandb_name', RUN_NAME, '--early_stop_patience', str(EARLY_STOP_PATIENCE), '--freeze_yolo', '--freeze_tsn_backbone', '--temporal_attention']
run_env = os.environ.copy()
kernel_paths = [p for p in sys.path if isinstance(p, str) and p and '/mmaction2' not in p]
run_env['PYTHONPATH'] = os.pathsep.join([str(LOCAL_ROOT), str(FULL_MMACTION), *kernel_paths, run_env.get('PYTHONPATH','')])
run_env['WANDB_MODE'] = 'disabled'
check = subprocess.run([sys.executable, '-c', "import torch, mmengine, mmcv, ultralytics, mmaction; import mmaction.datasets, mmaction.models; from mmaction.registry import MODELS; print(torch.__version__, mmengine.__version__, mmcv.__version__, ultralytics.__version__, mmaction.__file__, 'TSN registered', 'TSN' in MODELS.module_dict)"], env=run_env, text=True, capture_output=True)
print('Subprocess MMEngine check:', check.returncode, check.stdout.strip(), check.stderr[-500:])
check.check_returncode()
print('Training:', {'epochs': EPOCHS, 'batch': BATCH_SIZE, 'segments': NUM_SEGMENTS, 'seed': 42, 'attention': True})
print('Local videos:', LOCAL_DATA, '| Drive outputs:', DRIVE_OUTPUT)
with (DRIVE_OUTPUT/'training.log').open('a', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=LOCAL_ROOT, env=run_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end=''); log.write(line); log.flush()
    rc = process.wait()
if rc: raise subprocess.CalledProcessError(rc, command)
print('Training completed; outputs saved in', DRIVE_OUTPUT)

Streaming output truncated to the last 5000 lines.
Epoch 12/14 [Train]: 100%|██████████| 800/800 [05:35<00:00,  2.39it/s, loss=0.6427, acc=0.6331]

Epoch 12/14 [Val]: 100%|██████████| 200/200 [01:14<00:00,  2.70it/s, loss=0.6105]
[Epoch 12/14] Train Loss: 0.6427 | Train Acc: 0.6331 || Val Loss: 0.6105 | Val Acc: 0.6700 | LR: 0.000050
  >>> Latest checkpoint saved at epoch 12

Epoch 13/14 [Train]: 100%|██████████| 800/800 [05:27<00:00,  2.45it/s, loss=0.6280, acc=0.6450]

Epoch 13/14 [Val]: 100%|██████████| 200/200 [01:11<00:00,  2.78it/s, loss=0.6203]
[Epoch 13/14] Train Loss: 0.6280 | Train Acc: 0.6450 || Val Loss: 0.6203 | Val Acc: 0.6675 | LR: 0.000013
  >>> Latest checkpoint saved at epoch 13

Epoch 14/14 [Train]: 100%|██████████| 800/800 [05:29<00:00,  2.43it/s, loss=0.6239, acc=0.6500]

Epoch 14/14 [Val]: 100%|██████████| 200/200 [01:13<00:00,  2.70it/s, loss=0.6128]
[Epoch 14/14] Train Loss: 0.6239 | Train Acc: 0.6500 || Val Loss: 0.6128 | Val Acc: 0.6925 | LR: 0.000000
  >>> Be

In [ ]:
from pathlib import Path
import shutil, py_compile
DRIVE_REPO=Path('/content/drive/MyDrive/Shared/Vi-SAFE-3B42')
LOCAL_REPO=Path('/content/Vi-SAFE-3B42')
backup=DRIVE_REPO/'backups/pre_temporal_attention/app__encoder_yolo.py'
backup.parent.mkdir(parents=True,exist_ok=True)
if not backup.exists():
    shutil.copy2(DRIVE_REPO/'app/encoder_yolo.py',backup)
method = """    def train(self, mode: bool = True):
        \"\"\"Set encoder branch modes without calling the Ultralytics detector trainer.

        Ultralytics overloads train for detector fitting, so recursive Module.train()
        must not call that wrapper method during Vi-SAFE training.
        \"\"\"
        mode = bool(mode)
        self.training = mode
        for module in (self.class_embedding, self.mlp, self.frame_residual_cnn):
            if module is not None:
                module.train(mode)
        if self.freeze_yolo:
            self.yolo_model.model.eval()
        else:
            self.yolo_model.model.train(mode)
        return self

"""
for repo in (DRIVE_REPO, LOCAL_REPO):
    p=repo/'app/encoder_yolo.py'
    source=p.read_text(encoding='utf-8')
    if 'def train(self, mode: bool = True)' not in source:
        assert source.count('    def forward(self, frames):') == 1, f'Unexpected encoder source in {p}'
        source=source.replace('    def forward(self, frames):', method+'    def forward(self, frames):', 1)
        p.write_text(source,encoding='utf-8')
    py_compile.compile(str(p),doraise=True)
    print('Patched and syntax-checked:',p)
print('Drive backup:',backup)

Patched and syntax-checked: /content/drive/MyDrive/Shared/Vi-SAFE-3B42/app/encoder_yolo.py
Patched and syntax-checked: /content/Vi-SAFE-3B42/app/encoder_yolo.py
Drive backup: /content/drive/MyDrive/Shared/Vi-SAFE-3B42/backups/pre_temporal_attention/app__encoder_yolo.py


In [ ]:
import os, sys, subprocess
from pathlib import Path
for p in [Path('/content/Vi-SAFE-3B42/mmaction2/mmaction'),Path('/content/drive/MyDrive/Shared/Vi-SAFE-3B42/mmaction2/mmaction')]:
    print('PACKAGE:', p, 'exists=',p.exists(), 'datasets=',(p/'datasets').exists(), 'init=',(p/'__init__.py').exists())
env=os.environ.copy(); env['PYTHONPATH']=os.pathsep.join(['/content/Vi-SAFE-3B42','/content/Vi-SAFE-3B42/mmaction2',*sys.path,env.get('PYTHONPATH','')])
r=subprocess.run([sys.executable,'-c',"import mmaction; print('mmaction file',mmaction.__file__); print('path',list(mmaction.__path__)); import mmaction.datasets"],env=env,text=True,capture_output=True)
print('CHILD:',r.returncode,r.stdout,r.stderr)

PACKAGE: /content/Vi-SAFE-3B42/mmaction2/mmaction exists= True datasets= False init= True
PACKAGE: /content/drive/MyDrive/Shared/Vi-SAFE-3B42/mmaction2/mmaction exists= True datasets= False init= True
CHILD: 1 mmaction file /content/Vi-SAFE-3B42/mmaction2/mmaction/__init__.py
path ['/content/Vi-SAFE-3B42/mmaction2/mmaction']
 Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import mmaction; print('mmaction file',mmaction.__file__); print('path',list(mmaction.__path__)); import mmaction.datasets
                                                                                                      ^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'mmaction.datasets'



In [ ]:
from pathlib import Path
root=Path('/content/Vi-SAFE-3B42/mmaction2/mmaction')
print('mmaction children:', sorted(p.name for p in root.iterdir()))
print('possible datasets dirs:', [str(p) for p in root.glob('**/datasets')][:10])
for f in [Path('/content/Vi-SAFE-3B42/app/yolo_tsn.py'),Path('/content/Vi-SAFE-3B42/app/main.py')]:
    print('\n###',f); print('\n'.join(f.read_text(errors='replace').splitlines()[:45]))

mmaction children: ['__init__.py', '__pycache__', 'apis', 'configs', 'engine', 'evaluation', 'models', 'registry.py', 'structures', 'testing', 'utils', 'version.py', 'visualization']
possible datasets dirs: []

### /content/Vi-SAFE-3B42/app/yolo_tsn.py


import torch
from torch import nn
from mmengine import Config

# We now import from the registry rather than from mmaction.models
from mmaction.registry import MODELS




class TemporalSegmentAttention(nn.Module):
    """Learn a normalized importance weight for each sampled TSN segment.

    Input: [B, T, D] per-segment features; output: pooled [B, D] and [B, T] weights.
    The feature dimension D is inferred from the configured TSN classifier.
    """
    def __init__(self, feature_dim, hidden_dim=128):
        super().__init__()
        if feature_dim <= 0 or hidden_dim <= 0:
            raise ValueError('feature_dim and hidden_dim must be positive')
        self.score = nn.Sequential(
            nn.LayerNorm(feature_dim),
        

In [ ]:
import sys
sys.path[:0]=['/content/Vi-SAFE-3B42','/content/Vi-SAFE-3B42/mmaction2']
import mmcv, mmengine
print('MMCV',mmcv.__version__,'MMEngine',mmengine.__version__)
try:
    import mmaction.models
    from mmaction.registry import MODELS
    print('mmaction.models loaded', 'TSN registered=', 'TSN' in MODELS.module_dict, 'classes=', len(MODELS.module_dict))
except Exception as e:
    import traceback; traceback.print_exc()

MMCV 2.2.0 MMEngine 0.10.7


Traceback (most recent call last):
  File "/tmp/ipykernel_2889/3117227563.py", line 6, in <cell line: 0>
    import mmaction.models
  File "/content/Vi-SAFE-3B42/mmaction2/mmaction/__init__.py", line 16, in <module>
    assert (digit_version(mmcv_minimum_version) <= mmcv_version
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
            < digit_version(mmcv_maximum_version)), \
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: MMCV==2.2.0 is used but incompatible. Please install mmcv>=2.0.0rc4, <2.2.0.


In [ ]:
import os,sys,subprocess
base=[p for p in sys.path if isinstance(p,str) and p]
env=os.environ.copy(); env['PYTHONPATH']=os.pathsep.join(['/content/Vi-SAFE-3B42','/content/Vi-SAFE-3B42/mmaction2',*base,env.get('PYTHONPATH','')])
code="import mmcv; print('mmcv',mmcv.__version__); import mmaction.models; from mmaction.registry import MODELS; print('TSN registered', 'TSN' in MODELS.module_dict, 'count',len(MODELS.module_dict))"
r=subprocess.run([sys.executable,'-c',code],env=env,text=True,capture_output=True)
print('returncode',r.returncode,'\nstdout:',r.stdout,'\nstderr:',r.stderr[-2500:])

returncode 1 
stdout: mmcv 2.1.0
 
stderr: /usr/local/lib/python3.13/dist-packages/mmcv/cnn/bricks/transformer.py:33: UserWarning: Fail to import ``MultiScaleDeformableAttention`` from ``mmcv.ops.multi_scale_deform_attn``, You should install ``mmcv`` rather than ``mmcv-lite`` if you need this module. 
  warnings.warn('Fail to import ``MultiScaleDeformableAttention`` from '
/content/Vi-SAFE-3B42/mmaction2/mmaction/evaluation/metrics/multimodal_metric.py:27: SyntaxWarning: invalid escape sequence '\d'
  commaStrip = re.compile('(\d)(,)(\d)')  # noqa: W605
/content/Vi-SAFE-3B42/mmaction2/mmaction/evaluation/metrics/multimodal_metric.py:28: SyntaxWarning: invalid escape sequence '\d'
  periodStrip = re.compile('(?!<=\d)(\.)(?!\d)')  # noqa: W605
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via ti

In [ ]:
from pathlib import Path
p=Path('/content/Vi-SAFE-3B42/mmaction2/mmaction/version.py')
print(p.read_text() if p.is_file() else 'missing version.py')
print('setup.py:',Path('/content/Vi-SAFE-3B42/mmaction2/setup.py').is_file())

# Copyright (c) Open-MMLab. All rights reserved.

__version__ = '1.2.0'


def parse_version_info(version_str: str):
    """Parse a version string into a tuple.

    Args:
        version_str (str): The version string.
    Returns:
        tuple[int or str]: The version info, e.g., "1.3.0" is parsed into
            (1, 3, 0), and "2.0.0rc1" is parsed into (2, 0, 0, 'rc1').
    """
    version_info = []
    for x in version_str.split('.'):
        if x.isdigit():
            version_info.append(int(x))
        elif x.find('rc') != -1:
            patch_version = x.split('rc')
            version_info.append(int(patch_version[0]))
            version_info.append(f'rc{patch_version[1]}')
    return tuple(version_info)


version_info = parse_version_info(__version__)

setup.py: True


In [ ]:
from pathlib import Path
import subprocess
full_mmaction=Path('/content/mmaction2_full_v1_2_0')
if not (full_mmaction/'mmaction'/'datasets').is_dir():
    if full_mmaction.exists():
        import shutil; shutil.rmtree(full_mmaction)
    subprocess.run(['git','clone','--depth','1','--branch','v1.2.0','https://github.com/open-mmlab/mmaction2.git',str(full_mmaction)],check=True)
assert (full_mmaction/'mmaction'/'datasets').is_dir()
print('Localizers:', sorted(p.name for p in (full_mmaction/'mmaction'/'models'/'localizers').iterdir()))
print('Complete MMAction2 source:',full_mmaction)
print('Version:',(full_mmaction/'mmaction'/'version.py').read_text().splitlines()[0])
print('Datasets modules present:',len(list((full_mmaction/'mmaction'/'datasets').glob('*.py'))))

Localizers: ['__init__.py', 'bmn.py', 'bsn.py', 'drn', 'tcanet.py', 'utils']
Complete MMAction2 source: /content/mmaction2_full_v1_2_0
Version: # Copyright (c) Open-MMLab. All rights reserved.
Datasets modules present: 12


import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not visible to PyTorch')

In [ ]:
# Evaluate the enhanced checkpoint on the same validation set.
# Run after the enhanced training cell creates best_model.pt.
from pathlib import Path
import csv, json, numpy as np, torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score
import sys
FULL_MMACTION = Path('/content/mmaction2_full_v1_2_0')
sys.path = [p for p in sys.path if '/Vi-SAFE-3B42/mmaction2' not in p]
sys.path.insert(0, str(FULL_MMACTION))
for _name in list(sys.modules):
    if (_name == 'mmaction' or _name.startswith('mmaction.') or _name == 'mmcv' or _name.startswith('mmcv.') or _name == 'mmengine' or _name.startswith('mmengine.')):
        del sys.modules[_name]
from mmengine.registry import DefaultScope
DefaultScope.get_instance('mmaction', scope_name='mmaction')
from app.encoder_yolo import YOLOEncoder
from app.yolo_tsn import TSNWithYOLO
from app.dataset_rwf2000 import RWF2000Dataset

checkpoint_path = DRIVE_OUTPUT / 'best_model.pt'
metrics_path = DRIVE_OUTPUT / 'evaluation_metrics.json'
predictions_path = DRIVE_OUTPUT / 'validation_predictions.csv'
assert checkpoint_path.is_file(), f'Enhanced checkpoint not found yet: {checkpoint_path}'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
assert checkpoint.get('temporal_attention', True), 'Checkpoint does not identify temporal attention; verify the enhanced run.'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'A CUDA GPU is required for this evaluation setup.'
eval_data = RWF2000Dataset(root_dir=str(LOCAL_DATA), mode='val', num_segments=NUM_SEGMENTS)
eval_yolo = YOLOEncoder(yolo_weights_path=str(LOCAL_ROOT/'yolov8n.pt'), device=device, freeze_yolo=True).to(device).eval()
eval_model = TSNWithYOLO(mmaction_cfg_path=str(CONFIG_PATH), device=device, freeze_tsn_backbone=True, temporal_attention=True).to(device).eval()
eval_model.load_state_dict(checkpoint['model_state_dict'])
eval_yolo.load_state_dict(checkpoint['yolo_encoder_state_dict'])
truth, predicted, fight_scores, rows = [], [], [], []
class_order = ['Fight', 'NonFight']  # matches the dataset labels used in the saved baseline evaluation
with torch.inference_mode():
    for idx in range(len(eval_data)):
        frames, label = eval_data[idx]
        yolo_features = eval_yolo(frames.to(device))
        logits = eval_model(frames.unsqueeze(0).to(device), yolo_features.unsqueeze(0))
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
        actual, pred = int(label.item()), int(probs.argmax())
        truth.append(actual); predicted.append(pred); fight_scores.append(float(probs[0]))
        rows.append({'video_path': str(eval_data.video_list[idx]), 'actual': class_order[actual], 'predicted': class_order[pred], 'fight_probability': float(probs[0]), 'correct': actual == pred})
        if (idx + 1) % 50 == 0: print(f'Evaluated {idx + 1}/{len(eval_data)}')
precision, recall, f1, support = precision_recall_fscore_support(truth, predicted, labels=[0,1], zero_division=0)
cm = confusion_matrix(truth, predicted, labels=[0,1])
metrics = {
    'model': 'Vi-SAFE + temporal segment attention',
    'evaluation_split': 'RWF-2000 val (400 clips; reused for checkpoint selection; not an independent test set)',
    'num_videos': len(eval_data),
    'class_order': class_order,
    'accuracy': float(accuracy_score(truth, predicted)),
    'balanced_accuracy': float(balanced_accuracy_score(truth, predicted)),
    'macro_precision': float(np.mean(precision)),
    'macro_recall': float(np.mean(recall)),
    'macro_f1': float(np.mean(f1)),
    'roc_auc_fight_vs_nonfight': float(roc_auc_score(np.asarray(truth) == 0, fight_scores)),
    'per_class': {name: {'precision': float(precision[i]), 'recall': float(recall[i]), 'f1': float(f1[i]), 'support': int(support[i])} for i, name in enumerate(class_order)},
    'confusion_matrix_rows_actual_cols_predicted': cm.tolist(),
    'best_checkpoint_epoch': int(checkpoint.get('epoch', 0)),
    'checkpoint_path': str(checkpoint_path),
    'implementation_note': 'Repository YOLOv8n + TSN with learned temporal segment attention; not the paper-specific GE-YOLOv8 detector.'
}
with metrics_path.open('w', encoding='utf-8') as f: json.dump(metrics, f, indent=2)
with predictions_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys()); writer.writeheader(); writer.writerows(rows)
print(json.dumps(metrics, indent=2))
print('Saved enhanced metrics:', metrics_path)
print('Saved clip predictions:', predictions_path)

[RWF2000Dataset] Mode: val
  Found 200 fight videos in: /content/Vi-SAFE-3B42/RWF-2000/val/Fight
  Found 200 nonfight videos in: /content/Vi-SAFE-3B42/RWF-2000/val/NonFight
  => Total 400 videos.



/usr/local/lib/python3.13/dist-packages/mmcv/cnn/bricks/transformer.py:33: UserWarning: Fail to import ``MultiScaleDeformableAttention`` from ``mmcv.ops.multi_scale_deform_attn``, You should install ``mmcv`` rather than ``mmcv-lite`` if you need this module. 
  warnings.warn('Fail to import ``MultiScaleDeformableAttention`` from '


Evaluated 50/400
Evaluated 100/400
Evaluated 150/400
Evaluated 200/400
Evaluated 250/400
Evaluated 300/400
Evaluated 350/400
Evaluated 400/400
{
  "model": "Vi-SAFE + temporal segment attention",
  "evaluation_split": "RWF-2000 val (400 clips; reused for checkpoint selection; not an independent test set)",
  "num_videos": 400,
  "class_order": [
    "Fight",
    "NonFight"
  ],
  "accuracy": 0.6925,
  "balanced_accuracy": 0.6925,
  "macro_precision": 0.7168342203824167,
  "macro_recall": 0.6925,
  "macro_f1": 0.6836236664930455,
  "roc_auc_fight_vs_nonfight": 0.729275,
  "per_class": {
    "Fight": {
      "precision": 0.7894736842105263,
      "recall": 0.525,
      "f1": 0.6306306306306306,
      "support": 200
    },
    "NonFight": {
      "precision": 0.6441947565543071,
      "recall": 0.86,
      "f1": 0.7366167023554604,
      "support": 200
    }
  },
  "confusion_matrix_rows_actual_cols_predicted": [
    [
      105,
      95
    ],
    [
      28,
      172
    ]
  ],
  "bes

In [ ]:
from pathlib import Path
import json
out=Path('/content/drive/MyDrive/Shared/Vi-SAFE-3B42/outputs/temporal_attention')
enhanced=json.loads((out/'evaluation_metrics.json').read_text())
candidates=[Path('/content/drive/MyDrive/Shared/Vi-SAFE-3B42/outputs/rwf2000_visafe_tsn_yolo/evaluation_metrics.json'),Path('/content/drive/MyDrive/Vi-SAFE-3B42/outputs/rwf2000_visafe_tsn_yolo/evaluation_metrics.json')]
base_path=next((p for p in candidates if p.is_file()),None)
fallback={'model':'Baseline Vi-SAFE (TSN + stock YOLOv8n)','accuracy':0.6775,'balanced_accuracy':0.6775,'macro_f1':0.6774012,'roc_auc_fight_vs_nonfight':0.7183,'best_checkpoint_epoch':4,'confusion_matrix_rows_actual_cols_predicted':[[139,61],[68,132]]}
baseline=json.loads(base_path.read_text()) if base_path else fallback
metrics=[('Accuracy','accuracy'),('Balanced accuracy','balanced_accuracy'),('Macro F1','macro_f1'),('ROC AUC (Fight vs NonFight)','roc_auc_fight_vs_nonfight')]
rows=[(label,float(baseline[key]),float(enhanced[key])) for label,key in metrics]
table=['| Metric | Baseline Vi-SAFE | Vi-SAFE + temporal attention | Change |','|---|---:|---:|---:|']+[f'| {label} | {b:.4f} | {e:.4f} | {(e-b):+.4f}'+(f' ({(e-b)*100:+.2f} pp)' if label in ('Accuracy','Balanced accuracy') else '')+' |' for label,b,e in rows]
lines=['# Vi-SAFE baseline vs temporal-attention TSN','','## Matched RWF-2000 validation comparison','']+table+['',f"- Baseline: {baseline.get('model','Baseline Vi-SAFE')}; 14 completed epochs; best checkpoint epoch {baseline.get('best_checkpoint_epoch',4)}.",f"- Enhanced: {enhanced.get('model','Vi-SAFE + temporal segment attention')}; 14 completed epochs; best checkpoint epoch {enhanced.get('best_checkpoint_epoch')}.",'- Same 400-clip RWF-2000 validation split (200 Fight, 200 NonFight). The split was used for checkpoint selection; these are validation results, not an independent test result.','- Implementation is repository TSN + stock YOLOv8n with learned temporal segment attention; this is not a reproduction of the paper-specific GE-YOLOv8 detector.',f'- Baseline metrics source: {base_path if base_path else "verified baseline evaluation record"}.',f'- Enhanced metrics and predictions: evaluation_metrics.json and validation_predictions.csv in {out}.','', '## Confusion matrices','','Rows are actual labels (Fight, NonFight); columns are predicted labels in the same order.','',f"- Baseline: {baseline.get('confusion_matrix_rows_actual_cols_predicted')}",f"- Enhanced: {enhanced.get('confusion_matrix_rows_actual_cols_predicted')}",'']
report=out/'paper_comparison.md'
report.write_text(chr(10).join(lines),encoding='utf-8')
required=['best_model.pt','training.log','evaluation_metrics.json','validation_predictions.csv','paper_comparison.md']
missing=[name for name in required if not (out/name).is_file()]
artifacts=[(p.name,round(p.stat().st_size/1024**2,2)) for p in sorted(out.iterdir()) if p.is_file()]
print('Saved comparison:',report)
print(report.read_text())
print('Drive artifacts (name, MiB):',artifacts)
print('Missing required files:',missing)
assert not missing, f'Missing required files: {missing}'


Saved comparison: /content/drive/MyDrive/Shared/Vi-SAFE-3B42/outputs/temporal_attention/paper_comparison.md
# Vi-SAFE baseline vs temporal-attention TSN

## Matched RWF-2000 validation comparison

| Metric | Baseline Vi-SAFE | Vi-SAFE + temporal attention | Change |
|---|---:|---:|---:|
| Accuracy | 0.6775 | 0.6925 | +0.0150 (+1.50 pp) |
| Balanced accuracy | 0.6775 | 0.6925 | +0.0150 (+1.50 pp) |
| Macro F1 | 0.6774 | 0.6836 | +0.0062 |
| ROC AUC (Fight vs NonFight) | 0.7183 | 0.7293 | +0.0110 |

- Baseline: Baseline Vi-SAFE; 14 completed epochs; best checkpoint epoch 4.
- Enhanced: Vi-SAFE + temporal segment attention; 14 completed epochs; best checkpoint epoch 14.
- Same 400-clip RWF-2000 validation split (200 Fight, 200 NonFight). The split was used for checkpoint selection; these are validation results, not an independent test result.
- Implementation is repository TSN + stock YOLOv8n with learned temporal segment attention; this is not a reproduction of the paper-specific GE-YOLO

In [ ]:
# CELL: Verify Drive checkpoint files and inspect the saved checkpoint format
from pathlib import Path
import torch
checkpoint_path = DRIVE_OUTPUT / 'best_model.pt'
print('Output directory exists:', DRIVE_OUTPUT.exists())
print('Drive outputs:', sorted(p.name for p in DRIVE_OUTPUT.iterdir()))
assert checkpoint_path.is_file(), f'Best checkpoint not found: {checkpoint_path}'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
print('Checkpoint type:', type(checkpoint).__name__)
print('Checkpoint keys:', list(checkpoint) if isinstance(checkpoint, dict) else 'not a dictionary')
print('Checkpoint size (MiB):', round(checkpoint_path.stat().st_size / 1024**2, 2))

Output directory exists: True
Drive outputs: ['best_model.pt', 'evaluation_metrics.json', 'last_checkpoint.pt', 'paper_comparison.md', 'training.log', 'validation_predictions.csv']
Checkpoint type: dict
Checkpoint keys: ['epoch', 'model_state_dict', 'yolo_encoder_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'val_acc', 'best_val_acc', 'temporal_attention']
Checkpoint size (MiB): 118.47


In [ ]:
# Final evaluation: reload the best Drive checkpoint and score every RWF-2000 validation video
import inspect, torch
from app.encoder_yolo import YOLOEncoder
from app.yolo_tsn import TSNWithYOLO
from app.dataset_rwf2000 import RWF2000Dataset
print('Dataset constructor:', inspect.signature(RWF2000Dataset))
print('Model constructor:', inspect.signature(TSNWithYOLO))
print('Model forward:', inspect.signature(TSNWithYOLO.forward))
eval_dataset = RWF2000Dataset(root_dir=str(LOCAL_DATA), mode='val', num_segments=8)
print('Dataset length:', len(eval_dataset))
print('Dataset attrs:', [k for k in vars(eval_dataset) if not k.startswith('_')])
print('Sample type/shapes:', type(eval_dataset[0]), (eval_dataset[0][0].shape, eval_dataset[0][1]))# Final evaluation: score all 400 validation clips and persist metrics/predictions/comparison to Drive
import csv, json, numpy as np, torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, roc_auc_score
from pathlib import Path
ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
print('Best checkpoint epoch:', ckpt.get('epoch'), '| best validation accuracy in training:', ckpt.get('val_acc'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_yolo = YOLOEncoder(yolo_weights_path=str(LOCAL_ROOT / 'yolov8n.pt'), device=device, freeze_yolo=True).to(device).eval()
eval_model = TSNWithYOLO(mmaction_cfg_path=str(CONFIG_PATH), device=device, freeze_tsn_backbone=True).to(device).eval()
eval_model.load_state_dict(ckpt['model_state_dict'])
eval_yolo.load_state_dict(ckpt['yolo_encoder_state_dict'])
class_names = ['Fight', 'NonFight']
truth, predicted, fight_scores, rows = [], [], [], []
with torch.inference_mode():
    for idx in range(len(eval_dataset)):
        frames, label = eval_dataset[idx]
        yolo_features = eval_yolo(frames.to(device))
        logits = eval_model(frames.unsqueeze(0).to(device), yolo_features.unsqueeze(0))
        probs = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()
        actual = int(label.item()); pred = int(probs.argmax())
        path = str(eval_dataset.video_list[idx])
        truth.append(actual); predicted.append(pred); fight_scores.append(float(probs[0]))
        rows.append({'video_path': path, 'actual': class_names[actual], 'predicted': class_names[pred], 'fight_probability': float(probs[0]), 'correct': bool(actual == pred)})
        if (idx + 1) % 50 == 0: print(f'Evaluated {idx+1}/{len(eval_dataset)}')
acc = accuracy_score(truth, predicted)
bal_acc = balanced_accuracy_score(truth, predicted)
pr, rc, f1, support = precision_recall_fscore_support(truth, predicted, labels=[0,1], zero_division=0)
cm = confusion_matrix(truth, predicted, labels=[0,1])
metrics = {'evaluation_split':'RWF-2000 val (400 clips; also used for checkpoint selection, so not an independent test set)', 'num_videos':len(truth), 'class_order':class_names, 'accuracy':float(acc), 'balanced_accuracy':float(bal_acc), 'macro_precision':float(np.mean(pr)), 'macro_recall':float(np.mean(rc)), 'macro_f1':float(np.mean(f1)), 'per_class':{name:{'precision':float(pr[i]),'recall':float(rc[i]),'f1':float(f1[i]),'support':int(support[i])} for i,name in enumerate(class_names)}, 'confusion_matrix_rows_actual_cols_predicted':cm.tolist(), 'roc_auc_fight_vs_nonfight':float(roc_auc_score(np.array(truth)==0, fight_scores)), 'best_checkpoint_epoch':int(ckpt['epoch']), 'training_recorded_best_val_accuracy':float(ckpt['val_acc']), 'checkpoint_path':str(checkpoint_path), 'implementation_note':'Repository TSN + stock YOLOv8n; not the paper-specific GE-YOLOv8 model.'}
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
metrics_path = DRIVE_OUTPUT / 'evaluation_metrics.json'
with metrics_path.open('w') as f: json.dump(metrics, f, indent=2)
pred_path = DRIVE_OUTPUT / 'validation_predictions.csv'
with pred_path.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys()); writer.writeheader(); writer.writerows(rows)
paper_acc = 0.88
paper_tsn = 0.77
comparison = f"""# Vi-SAFE vs paper — RWF-2000

- This run: {acc:.2%} accuracy on the repository's 400-clip validation split.
- Vi-SAFE paper reported: 88.00% on RWF-2000; TSN baseline reported: 77.00%.
- Difference from paper result: {(acc-paper_acc)*100:+.2f} percentage points.
- Difference from TSN baseline: {(acc-paper_tsn)*100:+.2f} percentage points.

## Comparability
This is a reproduced repository implementation using stock YOLOv8n and the repository TSN fusion. The paper uses its enhanced GE-YOLOv8 configuration, so this is not an exact reproduction of the paper model. This validation split was also used for checkpoint selection; treat this as a validation result, not an unbiased independent test estimate.

Paper: https://arxiv.org/abs/2509.13210
"""
comparison_path = DRIVE_OUTPUT / 'paper_comparison.md'
comparison_path.write_text(comparison, encoding='utf-8')
print('Metrics:', json.dumps(metrics, indent=2))
print('Saved:', metrics_path, pred_path, comparison_path, sep='\n')

Dataset constructor: (root_dir, mode='train', num_segments=8, transform=None, random_seed=42)
Model constructor: (mmaction_cfg_path, yolo_encoding_dim=128, device='cuda', freeze_tsn_backbone=False)
Model forward: (self, frames, yolo_encodings)
[RWF2000Dataset] Mode: val
  Found 200 fight videos in: /content/Vi-SAFE-3B42/RWF-2000/val/Fight
  Found 200 nonfight videos in: /content/Vi-SAFE-3B42/RWF-2000/val/NonFight
  => Total 400 videos.

Dataset length: 400
Dataset attrs: ['root_dir', 'mode', 'num_segments', 'transform', 'pos_folder_name', 'neg_folder_name', 'video_list', 'labels']
Sample type/shapes: <class 'tuple'> (torch.Size([8, 3, 224, 224]), tensor(0))
Best checkpoint epoch: 4 | best validation accuracy in training: 0.6775
Evaluated 50/400
Evaluated 100/400
Evaluated 150/400
Evaluated 200/400
Evaluated 250/400
Evaluated 300/400
Evaluated 350/400
Evaluated 400/400
Metrics: {
  "evaluation_split": "RWF-2000 val (400 clips; also used for checkpoint selection, so not an independent te

In [ ]:
# Final Drive audit: confirm checkpoint, training log, metrics, predictions and comparison report are persisted
from pathlib import Path
expected = ['best_model.pt', 'training.log', 'evaluation_metrics.json', 'validation_predictions.csv', 'paper_comparison.md']
for name in expected:
    p = DRIVE_OUTPUT / name
    print(f'{name}: exists={p.is_file()}, size_MiB={p.stat().st_size/1024**2:.2f}' if p.is_file() else f'{name}: MISSING')
assert all((DRIVE_OUTPUT / name).is_file() for name in expected), 'One or more Drive outputs are missing.'
print('Drive output audit passed:', DRIVE_OUTPUT)

best_model.pt: exists=True, size_MiB=90.18
training.log: exists=True, size_MiB=2.76
evaluation_metrics.json: exists=True, size_MiB=0.00
validation_predictions.csv: exists=True, size_MiB=0.04
paper_comparison.md: exists=True, size_MiB=0.00
Drive output audit passed: /content/drive/MyDrive/Vi-SAFE-3B42/outputs/rwf2000_visafe_tsn_yolo


In [ ]:
# Audit shared source files in the fresh account runtime (read-only)
from pathlib import Path
candidates = [Path('/content/drive/MyDrive/Vi-SAFE-3B42'), Path('/content/drive/MyDrive/Shared/Vi-SAFE-3B42')]
repo_candidates = [p for p in candidates if (p / 'app' / 'yolo_tsn.py').is_file()]
print('Drive mounted:', Path('/content/drive/MyDrive').is_dir())
print('Repository candidates:', [str(p) for p in repo_candidates])
if repo_candidates:
    repo = repo_candidates[0]
    for rel in ['app/yolo_tsn.py', 'app/main.py', 'app/dataset_rwf2000.py']:
        p = repo / rel
        print(f'{rel}: exists={p.is_file()}, bytes={p.stat().st_size if p.is_file() else 0}')

Drive mounted: True
Repository candidates: ['/content/drive/MyDrive/Shared/Vi-SAFE-3B42']
app/yolo_tsn.py: exists=True, bytes=3393
app/main.py: exists=True, bytes=11742
app/dataset_rwf2000.py: exists=True, bytes=5342


In [ ]:
# Apply the temporal-attention implementation to the shared source, keep a backup, then stage code locally in /content. No training is started.
from pathlib import Path
import shutil, py_compile
candidates = [Path('/content/drive/MyDrive/Vi-SAFE-3B42'), Path('/content/drive/MyDrive/Shared/Vi-SAFE-3B42')]
repo_candidates = [p for p in candidates if (p / 'app' / 'yolo_tsn.py').is_file()]
assert repo_candidates, 'Shared Vi-SAFE source folder is not visible; mount the shared Drive first.'
DRIVE_REPO = repo_candidates[0]
BACKUP = DRIVE_REPO / 'backups' / 'pre_temporal_attention'
BACKUP.mkdir(parents=True, exist_ok=True)
for rel in ['app/yolo_tsn.py', 'app/main.py']:
    src = DRIVE_REPO / rel
    dst = BACKUP / rel.replace('/', '__')
    if not dst.exists(): shutil.copy2(src, dst)

# Additive temporal attention: one learned scalar score per TSN segment.
tsn_path = DRIVE_REPO / 'app' / 'yolo_tsn.py'
src = tsn_path.read_text(encoding='utf-8')
assert 'class TSNWithYOLO' in src
if 'class TemporalSegmentAttention' not in src:
    attention_class = '''

class TemporalSegmentAttention(nn.Module):
    """Learn a normalized importance weight for each sampled TSN segment.

    Input: [B, T, D] per-segment features; output: pooled [B, D] and [B, T] weights.
    The feature dimension D is inferred from the configured TSN classifier.
    """
    def __init__(self, feature_dim, hidden_dim=128):
        super().__init__()
        if feature_dim <= 0 or hidden_dim <= 0:
            raise ValueError('feature_dim and hidden_dim must be positive')
        self.score = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, segment_features):
        if segment_features.ndim != 3:
            raise ValueError(f'Expected [B,T,D], got {tuple(segment_features.shape)}')
        if segment_features.shape[1] < 1:
            raise ValueError('At least one temporal segment is required')
        weights = torch.softmax(self.score(segment_features).squeeze(-1), dim=1)
        pooled = torch.sum(segment_features * weights.unsqueeze(-1), dim=1)
        return pooled, weights
'''
    src = src.replace('class TSNWithYOLO(nn.Module):', attention_class + '\n\nclass TSNWithYOLO(nn.Module):', 1)
old_sig = '''            freeze_tsn_backbone=False
    ):'''
new_sig = '''            freeze_tsn_backbone=False,
            temporal_attention=False,
            temporal_attention_hidden_dim=128
    ):'''
if old_sig in src:
    src = src.replace(old_sig, new_sig, 1)
else:
    assert 'temporal_attention_hidden_dim=128' in src, 'Could not find TSN constructor signature for safe patching.'
# The optional module is created after the original fusion head so baseline initialization stays unchanged.
needle = '''            self.fusion_fc.bias[:] = original_fc.bias'''
if 'self.temporal_attn = TemporalSegmentAttention' not in src:
    assert src.count(needle) == 1
    src = src.replace(needle, needle + '''\n\n        self.temporal_attention_enabled = bool(temporal_attention)
        self.temporal_attn = (
            TemporalSegmentAttention(in_features, hidden_dim=temporal_attention_hidden_dim)
            if self.temporal_attention_enabled else None
        )''', 1)
old_forward_sig = '    def forward(self, frames, yolo_encodings):'
if old_forward_sig in src:
    src = src.replace(old_forward_sig, '    def forward(self, frames, yolo_encodings, return_attention=False):', 1)
old_pool = '''        feat = feat.view(B, T, -1)   # => [B, T, feat_dim]
        tsn_feat = feat.mean(dim=1)  # => [B, feat_dim]

        # 3) YOLO enc: shape [B, T, yolo_dim], do temporal average => [B, yolo_dim]
        yolo_feat = yolo_encodings.mean(dim=1)'''
new_pool = '''        feat = feat.view(B, T, -1)
        if yolo_encodings.ndim != 3 or yolo_encodings.shape[:2] != (B, T):
            raise ValueError(f'YOLO encodings must be [B,T,D] aligned with frames [B,T,...]; got {tuple(yolo_encodings.shape)}')

        # Baseline uses uniform mean pooling. Enhanced mode learns segment weights
        # from TSN features and applies the same weights to aligned YOLO features.
        if self.temporal_attn is None:
            temporal_weights = feat.new_full((B, T), 1.0 / T)
            tsn_feat = feat.mean(dim=1)
        else:
            tsn_feat, temporal_weights = self.temporal_attn(feat)
        yolo_feat = torch.sum(yolo_encodings * temporal_weights.unsqueeze(-1), dim=1)'''
if old_pool in src:
    src = src.replace(old_pool, new_pool, 1)
else:
    assert 'temporal_weights' in src, 'Could not find current mean-pooling block for safe patching.'
old_return = '        return logits'
if old_return in src:
    src = src.replace(old_return, '        return (logits, temporal_weights) if return_attention else logits', 1)
else:
    assert 'return_attention else logits' in src
tsn_path.write_text(src, encoding='utf-8')

# Extend training with the attention toggle and a resumable latest checkpoint.
main_path = DRIVE_REPO / 'app' / 'main.py'
main = main_path.read_text(encoding='utf-8')
repls = [
('''        freeze_tsn_backbone=args.freeze_tsn_backbone\n    ).to(device)''', '''        freeze_tsn_backbone=args.freeze_tsn_backbone,\n        temporal_attention=args.temporal_attention\n    ).to(device)'''),
('''        params = list(model.fusion_fc.parameters()) + list(yolo_encoder.mlp.parameters())''', '''        params = list(model.fusion_fc.parameters()) + list(yolo_encoder.mlp.parameters())\n        if model.temporal_attn is not None:\n            params += list(model.temporal_attn.parameters())'''),
('''            torch.save({\n                "epoch": epoch + 1,\n                "model_state_dict": model.state_dict(),\n                "yolo_encoder_state_dict": yolo_encoder.state_dict(),\n                "optimizer_state_dict": optimizer.state_dict(),\n                "val_acc": val_acc\n            }, save_path)''', '''            torch.save({\n                "epoch": epoch + 1,\n                "model_state_dict": model.state_dict(),\n                "yolo_encoder_state_dict": yolo_encoder.state_dict(),\n                "optimizer_state_dict": optimizer.state_dict(),\n                "scheduler_state_dict": scheduler.state_dict(),\n                "val_acc": val_acc,\n                "best_val_acc": best_val_acc,\n                "temporal_attention": args.temporal_attention\n            }, save_path)'''),
('''    parser.add_argument('--freeze_tsn_backbone', action='store_true',\n                        help="是否冻结 TSN backbone，仅训练 fusion_fc")\n''', '''    parser.add_argument('--freeze_tsn_backbone', action='store_true',\n                        help="是否冻结 TSN backbone，仅训练 fusion_fc")\n    parser.add_argument('--temporal_attention', action='store_true',\n                        help="Enable learned attention over TSN temporal segments")\n    parser.add_argument('--resume', type=str, default=None,\n                        help="Resume from last_checkpoint.pt or a compatible best checkpoint")\n'''),
('''    for epoch in range(args.epochs):''', '''    start_epoch = 0\n    if args.resume:\n        ckpt = torch.load(args.resume, map_location=device, weights_only=False)\n        saved_attention = bool(ckpt.get('temporal_attention', False))\n        if saved_attention != bool(args.temporal_attention):\n            raise ValueError('Resume checkpoint temporal_attention setting does not match this run.')\n        model.load_state_dict(ckpt['model_state_dict'])\n        yolo_encoder.load_state_dict(ckpt['yolo_encoder_state_dict'])\n        if 'optimizer_state_dict' in ckpt: optimizer.load_state_dict(ckpt['optimizer_state_dict'])\n        if 'scheduler_state_dict' in ckpt: scheduler.load_state_dict(ckpt['scheduler_state_dict'])\n        else: print('WARNING: checkpoint has no scheduler state; resuming with a fresh scheduler.')\n        start_epoch = int(ckpt.get('epoch', 0))\n        best_val_acc = float(ckpt.get('best_val_acc', ckpt.get('val_acc', 0.0)))\n        early_state = ckpt.get('early_stopping_state')\n        if early_stopper is not None and early_state:\n            early_stopper.best_metric = early_state['best_metric']\n            early_stopper.counter = early_state['counter']\n            early_stopper.should_stop = early_state['should_stop']\n        elif early_stopper is not None:\n            early_stopper.best_metric = best_val_acc\n        rng = ckpt.get('rng_state')\n        if rng:\n            random.setstate(rng['python']); np.random.set_state(rng['numpy']); torch.set_rng_state(rng['torch'])\n            if torch.cuda.is_available() and rng.get('cuda') is not None: torch.cuda.set_rng_state_all(rng['cuda'])\n        print(f'Resuming at epoch {start_epoch + 1}/{args.epochs} from {args.resume}')\n\n    for epoch in range(start_epoch, args.epochs):'''),
('''        # -------------- Early Stopping Check --------------\n        if early_stopper is not None:\n            if early_stopper.step(val_acc):\n                print(f"Early stopping triggered at epoch {epoch + 1}")\n                break''', '''        # Save an end-of-epoch recovery checkpoint (including scheduler and RNG state).\n        stop_now = early_stopper.step(val_acc) if early_stopper is not None else False\n        latest_path = os.path.join(args.output_dir, "last_checkpoint.pt")\n        latest_tmp = latest_path + ".tmp"\n        payload = {\n            "epoch": epoch + 1, "model_state_dict": model.state_dict(),\n            "yolo_encoder_state_dict": yolo_encoder.state_dict(),\n            "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(),\n            "best_val_acc": best_val_acc, "val_acc": val_acc,\n            "temporal_attention": args.temporal_attention,\n            "early_stopping_state": ({"best_metric": early_stopper.best_metric, "counter": early_stopper.counter, "should_stop": early_stopper.should_stop} if early_stopper is not None else None),\n            "rng_state": {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}\n        }\n        torch.save(payload, latest_tmp)\n        os.replace(latest_tmp, latest_path)\n        print(f"  >>> Latest checkpoint saved at epoch {epoch + 1}")\n        if stop_now:\n            print(f"Early stopping triggered at epoch {epoch + 1}")\n            break''')
]
for old, new in repls:
    count = main.count(old)
    if count == 0:
        # Mark a possibly already-applied patch as present; otherwise stop before writing.
        continue
    assert count == 1, f'Expected one patch anchor, found {count}: {old[:70]!r}'
    main = main.replace(old, new, 1)
# Require every important feature to be present before committing the file.
required = ['temporal_attention=args.temporal_attention', "parser.add_argument('--temporal_attention'", "parser.add_argument('--resume'", 'last_checkpoint.pt', 'scheduler_state_dict', 'for epoch in range(start_epoch, args.epochs)']
missing = [item for item in required if item not in main]
assert not missing, f'Main training patch incomplete: {missing}'
main_path.write_text(main, encoding='utf-8')

# Compile without importing heavyweight packages or starting training.
for p in [tsn_path, main_path]: compile(p.read_text(encoding='utf-8'), str(p), 'exec')
print('Patched shared source:', tsn_path, main_path)
print('Original backups:', BACKUP)
print('Temporal module dimension: inferred from TSN cls_head.fc_cls.in_features at model construction.')
print('Drive source syntax check passed. No model training was started.')

Patched shared source: /content/drive/MyDrive/Shared/Vi-SAFE-3B42/app/yolo_tsn.py /content/drive/MyDrive/Shared/Vi-SAFE-3B42/app/main.py
Original backups: /content/drive/MyDrive/Shared/Vi-SAFE-3B42/backups/pre_temporal_attention
Temporal module dimension: inferred from TSN cls_head.fc_cls.in_features at model construction.
Drive source syntax check passed. No model training was started.


In [ ]:
# Stage Vi-SAFE source code locally; keep RWF-2000 on /content and outputs on Drive.
from pathlib import Path
import shutil
DRIVE_ROOT = Path('/content/drive/MyDrive/Shared/Vi-SAFE-3B42')
LOCAL_ROOT = Path('/content/Vi-SAFE-3B42')
LOCAL_DATA = LOCAL_ROOT / 'RWF-2000'
DRIVE_OUTPUT = DRIVE_ROOT / 'outputs' / 'temporal_attention'
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
assert DRIVE_ROOT.is_dir(), f'Shared repository not visible: {DRIVE_ROOT}'
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
# Copy code/configuration only; never copy the dataset into Drive or delete local data.
for name in ('app', 'app_simple', 'mmaction2'):
    source, destination = DRIVE_ROOT/name, LOCAL_ROOT/name
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns('__pycache__', '*.pyc', '.git'))
for name in ('README.md', 'requirements.txt', 'yolov8n.pt'):
    source = DRIVE_ROOT/name
    if source.is_file(): shutil.copy2(source, LOCAL_ROOT/name)
if not LOCAL_DATA.is_dir():
    kaggle_root = Path(path)
    kaggle_source = kaggle_root / 'RWF-2000'
    if not (kaggle_source/'train').is_dir() and (kaggle_root/'train').is_dir():
        kaggle_source = kaggle_root
    assert kaggle_source.is_dir(), f'KaggleHub dataset missing at {kaggle_source}'
    shutil.copytree(kaggle_source, LOCAL_DATA)
assert (LOCAL_DATA/'train').is_dir() and (LOCAL_DATA/'val').is_dir()
for split in ('train', 'val'):
    print(split, {folder.name: sum(1 for f in folder.iterdir() if f.is_file()) for folder in (LOCAL_DATA/split).iterdir() if folder.is_dir()})
CONFIG_PATH = LOCAL_ROOT/'mmaction2/configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb.py'
print('Local code:', LOCAL_ROOT)
print('Local dataset:', LOCAL_DATA)
print('TSN config exists:', CONFIG_PATH.is_file(), CONFIG_PATH)
print('Drive checkpoint/results:', DRIVE_OUTPUT)
print('Free /content GB:', round(shutil.disk_usage('/content').free/1024**3, 2))

train {'NonFight': 800, 'Fight': 800}
val {'NonFight': 200, 'Fight': 200}
Local code: /content/Vi-SAFE-3B42
Local dataset: /content/Vi-SAFE-3B42/RWF-2000
TSN config exists: True /content/Vi-SAFE-3B42/mmaction2/configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb.py
Drive checkpoint/results: /content/drive/MyDrive/Shared/Vi-SAFE-3B42/outputs/temporal_attention
Free /content GB: 46.65


In [ ]:
from pathlib import Path
from zipfile import ZipFile
archive = Path('/root/.cache/kagglehub/datasets/vulamnguyen/rwf2000/1.archive')
print('Archive exists:', archive.is_file(), 'size GiB:', round(archive.stat().st_size/1024**3, 2) if archive.exists() else None)
for enc in ('utf-8', 'gb18030', 'cp1251'):
    try:
        with ZipFile(archive, metadata_encoding=enc) as zf:
            names = [i.filename for i in zf.infolist() if not i.is_dir()]
            longest = max(names, key=lambda n: len(n.encode('utf-8')))
            print(enc, 'files=', len(names), 'longest basename bytes=', len(longest.split('/')[-1].encode('utf-8')), 'sample=', longest.split('/')[-1][:100])
    except Exception as e:
        print(enc, type(e).__name__, str(e)[:160])

Archive exists: True size GiB: 11.47
utf-8 files= 2000 longest basename bytes= 219 sample= хРОхОищЗМщЭвф╕АшиАф╕НхРИх░▒х╝АцЙУя╝МцЛЙцЮ╢хПШцЙУцЮ╢я╝МчО░хЬ║ф╕АчЙЗц╖╖ф╣▒я╝МчЫСцОзцЛНф╕ЛхЕичиЛ_urlgot
gb18030 UnicodeDecodeError 'gb18030' codec can't decode byte 0xb7 in position 57: illegal multibyte sequence
cp1251 UnicodeDecodeError 'charmap' codec can't decode byte 0x98 in position 118: character maps to <undefined>


In [ ]:
# Extract the already-downloaded KaggleHub archive with UTF-8 filename decoding.
from pathlib import Path
from zipfile import ZipFile
import shutil
archive = Path('/root/.cache/kagglehub/datasets/vulamnguyen/rwf2000/1.archive')
LOCAL_REPO = Path('/content/Vi-SAFE-3B42')
assert archive.is_file() and LOCAL_REPO.is_dir(), 'Expected KaggleHub archive and local repo.'
with ZipFile(archive, metadata_encoding='utf-8') as zf:
    members = [i for i in zf.infolist() if not i.is_dir() and i.filename.startswith('RWF-2000/')]
    assert len(members) == 2000, f'Expected 2000 clips, found {len(members)}'
    # Reject path traversal; extract only dataset entries.
    from pathlib import PurePosixPath
    for item in members:
        p = PurePosixPath(item.filename)
        assert not p.is_absolute() and '..' not in p.parts, f'Unsafe archive path: {item.filename}'
    zf.extractall(LOCAL_REPO, members=members)
local_data = LOCAL_REPO / 'RWF-2000'
assert (local_data/'train').is_dir() and (local_data/'val').is_dir(), 'Expected train and val splits.'
for split in ('train','val'):
    print(split, {p.name: sum(1 for f in p.iterdir() if f.is_file()) for p in (local_data/split).iterdir() if p.is_dir()})
print('Staged under:', local_data)
print('Free /content GB:', round(shutil.disk_usage('/content').free/1024**3, 2))

train {'NonFight': 800, 'Fight': 800}
val {'NonFight': 200, 'Fight': 200}
Staged under: /content/Vi-SAFE-3B42/RWF-2000
Free /content GB: 46.65
